<a href="https://colab.research.google.com/github/adriansivina-droid/scraper-mef/blob/main/BI_Scraping_MEF_DEV_Funcion_GobiernoRegional_LOR_SMN_UCA_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ═══ BLOQUE 1 · INSTALACIÓN (ejecutar una vez por sesión) ═══
!pip install requests beautifulsoup4 pandas numpy tqdm google-api-python-client --quiet
print("✅ Librerías listas")
import os, re, time, random, zipfile, unicodedata
import requests
import numpy as np
import pandas as pd
from datetime import date
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from IPython.display import display

✅ Librerías listas


In [ ]:
# ═══ BLOQUE 2 · CONFIGURACIÓN Y FUNCIONES (no descarga nada) ═══
# ── 1. CONFIGURACIÓN ─────────────────────────────────────────────────────────
AÑO = 2025
MES_INICIO = 1                 # primer mes del bucle
MES_FIN = None                 # None = último mes disponible
INCLUIR_MES_EN_CURSO = False   # False = solo meses cerrados (regla del scraper anterior)

CONTROL_ANUAL = True           # compara contra la vista anual de Consulta Amigable
SUBIR_A_DRIVE = True           # sube cada CSV a la carpeta de Drive de arriba
DRIVE_FOLDER_ID = "18Ohcjmg6FojhNwbCHUHX3MEOxhokZA9b"
DESCARGA_FINAL = "ninguna"     # "zip" | "individual" | "ninguna"

# ── Nivel de detalle (equivale a "municipalidad" en el scraper municipal) ─────
#   "EJECUTORA"    -> Departamento > Pliego > Unidad Ejecutora > Función   (más consultas)
#   "PLIEGO"       -> Departamento > Pliego > Función
#   "DEPARTAMENTO" -> Departamento > Función
NIVEL_DETALLE = "EJECUTORA"

# ── Filtros de la ruta ───────────────────────────────────────────────────────
# Claves de la URL de Consulta Amigable. Un parámetro con valor VACÍO corta la
# ruta: el servidor ignora todo lo que viene después. Por eso el "37"
# (Gob.Loc./Mancom., que solo existe para municipalidades) NO va para regiones:
# con "37": "" la página se quedaba en "Nivel de Gobierno" y nunca filtraba
# el departamento (variante D de la celda de diagnóstico).
NIVEL_GOBIERNO = "R"                          # R = Gobiernos Regionales (M era Locales)
NIVEL_ETIQUETA = "R: GOBIERNOS REGIONALES"
GENERICA = "6-2-6"                            # Genérica 6-26
PARAMS_EXTRA = {}   # si F12>Network muestra otros parámetros, se sobreescriben aquí

PAUSA = (0.4, 1.0)             # segundos aleatorios entre consultas (cortesía con el servidor)

# ── Meses válidos ────────────────────────────────────────────────────────────
hoy = date.today()
if AÑO > hoy.year:
    raise ValueError("Año inválido: no puede ser futuro.")
if AÑO < hoy.year:
    meses_validos = list(range(1, 13))
else:
    tope = hoy.month if INCLUIR_MES_EN_CURSO else hoy.month - 1
    if tope < 1:
        raise ValueError("No hay meses cerrados aún en este año.")
    meses_validos = list(range(1, tope + 1))
ultimo_valido = meses_validos[-1]
if MES_FIN is None:
    MES_FIN = ultimo_valido
if not (1 <= MES_INICIO <= MES_FIN <= ultimo_valido):
    raise ValueError(f"Rango inválido ({MES_INICIO}-{MES_FIN}). Disponibles: 1 a {ultimo_valido}.")
meses = [str(m) for m in range(MES_INICIO, MES_FIN + 1)]

# ── Catálogos ────────────────────────────────────────────────────────────────
departamentos = [
    {"codigo": "16", "departamento": "16: LORETO"},
    {"codigo": "22", "departamento": "22: SAN MARTÍN"},
    {"codigo": "25", "departamento": "25: UCAYALI"},
]
nombres_meses = ["Enero","Febrero","Marzo","Abril","Mayo","Junio","Julio","Agosto",
                 "Setiembre","Octubre","Noviembre","Diciembre"]
meses_map  = {str(i): f"{i}: {nombres_meses[i-1]}" for i in range(1, 13)}
deptos_map = {d["codigo"]: d["departamento"] for d in departamentos}

# Botones de Consulta Amigable que se pulsan, en orden, debajo del departamento
RUTAS = {
    "EJECUTORA":    ["Pliego", "Ejecutora", "Función"],
    "PLIEGO":       ["Pliego", "Función"],
    "DEPARTAMENTO": ["Función"],
}
if NIVEL_DETALLE not in RUTAS:
    raise ValueError(f"NIVEL_DETALLE debe ser uno de {list(RUTAS)}")
BOTONES = RUTAS[NIVEL_DETALLE]
COLUMNA_DE = {"Pliego": "PLIEGO", "Ejecutora": "UNIDAD_EJECUTORA", "Función": "FUNCION"}

ruta_guardado = "/content/Inversion_Funcion.GobiernoRegional"
os.makedirs(ruta_guardado, exist_ok=True)
SUFIJO_REGIONES = "LOR - SMN - UCA"

def etiqueta_meses(ms):
    nums = sorted(int(m) for m in ms)
    if len(nums) == 1:
        return f"Mes {nums[0]:02d}"
    if nums == list(range(nums[0], nums[-1] + 1)):
        return f"Meses {nums[0]:02d}-{nums[-1]:02d}"
    return "Meses " + ",".join(f"{m:02d}" for m in nums)

def nombre_archivo_salida(año, ms):
    return f"DEV Funcion.GobRegional {año} - {etiqueta_meses(ms)} - {SUFIJO_REGIONES}.csv"

# ENTIDAD = "GOBIERNO REGIONAL DE <REGIÓN>" solo si no se baja a Pliego (sería la misma columna)
COLUMNAS_SALIDA = (["AÑO", "MES", "NIVEL_GOBIERNO", "DEPARTAMENTO"]
                   + ([] if "Pliego" in BOTONES else ["ENTIDAD"])
                   + [COLUMNA_DE[b] for b in BOTONES])
COLUMNAS_SALIDA += ["DEVENGADO"]
print(f"Año {AÑO} -> meses a procesar: {', '.join(meses)}  |  regiones: "
      f"{', '.join(d['departamento'] for d in departamentos)}")
print(f"Ruta por región: Departamento > {' > '.join(BOTONES)}")

# ── 2. GOOGLE DRIVE ──────────────────────────────────────────────────────────
drive_subidos, drive_fallidos = [], []
STATUS_REINTENTABLES = {403, 429, 500, 502, 503, 504}

def conectar_drive(folder_id):
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.errors import HttpError
    auth.authenticate_user()
    servicio = build("drive", "v3", cache_discovery=False)
    try:
        carpeta = servicio.files().get(
            fileId=folder_id, fields="id,name,mimeType,capabilities(canAddChildren)",
            supportsAllDrives=True).execute()
    except HttpError as e:
        raise PermissionError(f"Sin acceso a la carpeta de Drive (HTTP {e.resp.status}).") from e
    if carpeta.get("mimeType") != "application/vnd.google-apps.folder":
        raise ValueError("El ID no corresponde a una carpeta de Drive.")
    if not carpeta.get("capabilities", {}).get("canAddChildren", False):
        raise PermissionError(f"Sin permiso de Editor en '{carpeta['name']}'.")
    print(f"☁️  Drive conectado -> carpeta: '{carpeta['name']}'\n")
    return servicio

def subir_a_drive(servicio, ruta_local, folder_id, max_reintentos=5):
    from googleapiclient.errors import HttpError
    from googleapiclient.http import MediaFileUpload
    nombre = os.path.basename(ruta_local)
    mime = "application/zip" if nombre.lower().endswith(".zip") else "text/csv"
    nombre_q = nombre.replace("\\", "\\\\").replace("'", "\\'")
    ultimo_error = ""
    for intento in range(1, max_reintentos + 1):
        try:
            existentes = servicio.files().list(
                q=f"name = '{nombre_q}' and '{folder_id}' in parents and trashed = false",
                fields="files(id)", supportsAllDrives=True,
                includeItemsFromAllDrives=True).execute().get("files", [])
            media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)
            if existentes:
                servicio.files().update(fileId=existentes[0]["id"], media_body=media,
                                        fields="id", supportsAllDrives=True).execute()
                accion = "actualizado"
            else:
                servicio.files().create(body={"name": nombre, "parents": [folder_id]},
                                        media_body=media, fields="id",
                                        supportsAllDrives=True).execute()
                accion = "creado"
            drive_subidos.append(nombre)
            print(f"   ☁️ Drive ({accion}): {nombre}")
            return True
        except HttpError as e:
            ultimo_error = f"HTTP {e.resp.status}"
            if e.resp.status not in STATUS_REINTENTABLES:
                break
        except Exception as e:
            ultimo_error = str(e)
        if intento < max_reintentos:
            espera = 2 ** intento + random.random()
            print(f"   ↻ Reintento {intento}/{max_reintentos - 1} en {espera:.1f}s ({ultimo_error})")
            time.sleep(espera)
    drive_fallidos.append((nombre, ultimo_error))
    print(f"   ❌ No se pudo subir {nombre} ({ultimo_error}). Sigue en {ruta_local}")
    return False


# ── 3. NAVEGACIÓN EN CONSULTA AMIGABLE ───────────────────────────────────────
# La URL con parámetros solo arma el "breadcrumb" (TOTAL > Genérica > Mes >
# Nivel > Departamento). El listado de detalle (Función, Pliego, Ejecutora…)
# aparece recién al pulsar el botón correspondiente: un postback ASP.NET que
# reenvía los campos ocultos del formulario (__VIEWSTATE, etc.). Para bajar un
# nivel se marca la fila (radio "grp1") y se pulsa el siguiente botón.
URL_BASE = "https://apps5.mineco.gob.pe/transparencia/mensual/Navegar_6.aspx"
sesion = requests.Session()
sesion.headers.update({"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                                     "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"})

def a_numero(txt):
    t = (txt or "").replace(",", "").replace("\xa0", "").strip()
    if t in ("", "-"):
        return 0.0
    try:
        return float(t)
    except ValueError:
        return np.nan

def normalizar(txt):
    t = unicodedata.normalize("NFKD", txt or "")
    return "".join(c for c in t if not unicodedata.combining(c)).lower().strip()

_FILTRO = re.compile(r"^(total|gen[eé]rica|nivel de gobierno|gob\.?\s?loc|departamento|"
                     r"funci[oó]n|mes\b|trimestre|sector|pliego|ejecutora|unidad ejecutora|"
                     r"fuente|rubro|categor|producto|divisi[oó]n)", re.I)
_CODIGO = re.compile(r"^([\w\-\.]+)\s*:")

class Pagina:
    """Una respuesta de Consulta Amigable ya parseada.
    filtros: breadcrumb [(etiqueta, devengado)]
    filas  : detalle    [(valor_radio, etiqueta, devengado)]
    botones: {texto_normalizado: (name, value, deshabilitado)}"""
    def __init__(self, html, url):
        self.url, self.html = url, html
        s = BeautifulSoup(html, "html.parser")
        self.tiene_tabla = s.find("table", class_="Data") is not None
        form = s.find("form")
        self.action = urljoin(url, form.get("action") or url) if form else url
        self.campos, self.botones = {}, {}
        for inp in (form.find_all("input") if form else []):
            tipo = (inp.get("type") or "text").lower()
            name = inp.get("name")
            if not name:
                continue
            if tipo in ("submit", "button", "image"):
                self.botones[normalizar(inp.get("value"))] = (name, inp.get("value", ""),
                                                              inp.has_attr("disabled"))
            elif tipo in ("hidden", "text"):
                self.campos[name] = inp.get("value", "")
            elif tipo in ("checkbox",) and inp.has_attr("checked"):
                self.campos[name] = inp.get("value", "on")
        for sel in (form.find_all("select") if form else []):
            opt = sel.find("option", selected=True) or sel.find("option")
            if sel.get("name") and opt is not None:
                self.campos[sel["name"]] = opt.get("value", opt.get_text(strip=True))

        self.filtros, self.filas = [], []
        for tr in s.find_all("tr"):
            if tr.find("table"):
                continue
            celdas = tr.find_all("td")
            if len(celdas) < 8:
                continue
            dev = a_numero(celdas[-3].get_text(strip=True))   # … Devengado | Girado | Avance %
            if dev != dev:                                     # NaN -> no es fila numérica
                continue
            textos = [" ".join(c.get_text(" ", strip=True).split()) for c in celdas[:3]]
            etiqueta = next((t for t in textos if t), "")
            radio = tr.find("input", attrs={"type": "radio"})
            if _FILTRO.match(etiqueta):
                self.filtros.append((etiqueta, dev))
            elif radio is not None:
                self.filas.append((radio.get("value") or etiqueta, etiqueta, dev))
            elif _CODIGO.match(etiqueta):                      # detalle sin radio (por si acaso)
                self.filas.append((etiqueta, etiqueta, dev))

    def boton(self, texto):
        """Busca el botón por su texto (sin tildes ni mayúsculas)."""
        clave = normalizar(texto)
        if clave in self.botones:
            return self.botones[clave]
        candidatos = [v for k, v in self.botones.items() if k.startswith(clave) or clave in k]
        return candidatos[0] if candidatos else None

def _pedir(metodo, url, reintentos=4, **kw):
    ultimo = ""
    for i in range(reintentos):
        try:
            r = sesion.request(metodo, url, timeout=40, **kw)
            if r.status_code == 200:
                p = Pagina(r.content, r.url)
                if p.tiene_tabla:
                    return p, None
                ultimo = "SIN_TABLA"
            else:
                ultimo = f"HTTP {r.status_code}"
        except requests.RequestException as e:
            ultimo = f"{type(e).__name__}: {e}"
        time.sleep(min(30, 2 ** i + random.random()))
    return None, ultimo

def abrir_departamento(anio, mes, depto):
    """Página con breadcrumb TOTAL > Genérica > [Mes] > Nivel > Departamento.
    mes='' -> vista anual (se OMITE la clave 23: un valor vacío cortaría la ruta)."""
    params = {"0": "", "24": GENERICA}
    if mes:
        params["23"] = mes
    params.update({"1": NIVEL_GOBIERNO, "5": depto, "21": depto, "8": "", "7": "", "y": anio})
    params.update(PARAMS_EXTRA)
    params = {k: v for k, v in params.items() if v is not None}
    return _pedir("POST", URL_BASE, params=params)

def pulsar(pagina, texto_boton, seleccion=None):
    """Postback: marca la fila `seleccion` (radio grp1) y pulsa el botón."""
    b = pagina.boton(texto_boton)
    if b is None:
        disponibles = ", ".join(v[1] for v in pagina.botones.values()) or "ninguno"
        return None, f"no existe el botón '{texto_boton}' (botones: {disponibles})"
    name, value, deshabilitado = b
    if deshabilitado:
        return None, f"el botón '{texto_boton}' está deshabilitado en esta página"
    datos = dict(pagina.campos)
    datos[name] = value
    if seleccion is not None:
        datos["grp1"] = seleccion
    time.sleep(random.uniform(*PAUSA))
    return _pedir("POST", pagina.action, data=datos, headers={"Referer": pagina.url})

def cuadre(p, contexto):
    """Suma del detalle vs la última fila del breadcrumb (el filtro aplicado)."""
    if not p.filtros or not p.filas:
        return None
    total = p.filtros[-1][1]
    suma = sum(v for _, _, v in p.filas)
    if abs(suma - total) > max(1, len(p.filas)):
        return f"CUADRE {contexto}: suma {suma:,.0f} vs {p.filtros[-1][0][:40]} {total:,.0f}"
    return None

def validar_departamento(p, depto):
    errores = []
    etiquetas = [normalizar(e) for e, _ in p.filtros]
    if not any(e.startswith("nivel de gobierno") and "regional" in e for e in etiquetas):
        errores.append("la página no está filtrada por GOBIERNOS REGIONALES")
    if not any(e.startswith("departamento") and f"{depto}:" in e for e in etiquetas):
        errores.append(f"la página no corresponde al departamento {depto}")
    return errores

def recorrer(pagina, botones, seleccion=None, ruta=()):
    """Pulsa los botones en cadena. Devuelve (hojas, errores, avisos);
    hojas = [(ruta_de_etiquetas, codigo, etiqueta, devengado)] del último nivel."""
    p, err = pulsar(pagina, botones[0], seleccion)
    nombre = " > ".join(ruta) or "departamento"
    if err:
        return [], [f"{botones[0]} en {nombre}: {err}"], []
    if not p.filas:
        # sin detalle: válido si el filtro de arriba tiene devengado 0
        if p.filtros and p.filtros[-1][1] == 0:
            return [], [], []
        return [], [f"{botones[0]} en {nombre}: la página no trae filas de detalle"], []
    avisos = [a for a in [cuadre(p, f"{botones[0]} en {nombre}")] if a]
    if len(botones) == 1:
        hojas = [(ruta, (_CODIGO.match(et) or [None, et[:2]])[1], et, dev)
                 for _, et, dev in p.filas]
        return hojas, [], avisos
    hojas, errores = [], []
    for valor, etiqueta, dev in p.filas:
        if dev == 0:                       # nada que bajar
            continue
        h, e, a = recorrer(p, botones[1:], valor, ruta + (etiqueta,))
        hojas += h; errores += e; avisos += a
    return hojas, errores, avisos

def extraer(anio, mes, depto, botones=BOTONES):
    """-> (registros, error, aviso) para una región y un mes."""
    p, err = abrir_departamento(anio, mes, depto)
    if err:
        return [], err, None
    errores = validar_departamento(p, depto)
    if errores:
        return [], "VALIDACIÓN: " + "; ".join(errores), None
    hojas, errores, avisos = recorrer(p, botones)
    nombre_dep = deptos_map[depto].split(": ", 1)[1]
    regs = []
    for ruta, cod, etiqueta, dev in hojas:
        r = {"AÑO": anio, "MES": meses_map.get(mes, "ANUAL"), "NIVEL_GOBIERNO": NIVEL_ETIQUETA,
             "DEPARTAMENTO": deptos_map[depto], "COD_DEPTO": depto, "COD_FUNCION": cod,
             "DEVENGADO": dev}
        for b, valor in zip(botones[:-1], ruta):
            r[COLUMNA_DE[b]] = valor
        r[COLUMNA_DE[botones[-1]]] = etiqueta
        r["ENTIDAD"] = r.get("PLIEGO", f"GOBIERNO REGIONAL DE {nombre_dep}")
        regs.append(r)
    err = ("; ".join(errores)) if errores else None
    return regs, err, ("; ".join(avisos) if avisos else None)

# ── 4. CONTROL ANUAL (año x departamento x función x devengado) ──────────────
def control_anual(df_total, anio, mes_ultimo):
    """Compara lo raspado contra la vista anual de Consulta Amigable (sin filtro de mes).
    Detecta además si el Devengado de cada mes es MENSUAL (la suma de meses cuadra)
    o ACUMULADO (el último mes cuadra)."""
    filas = []
    for d in departamentos:
        regs, err, _ = extraer(anio, "", d["codigo"], botones=["Función"])
        if err or not regs:
            print(f"   ❌ Referencia anual {d['departamento']}: {err or 'sin filas'}")
            continue
        ref = {r["COD_FUNCION"]: r["DEVENGADO"] for r in regs}
        sub = df_total[df_total["COD_DEPTO"] == d["codigo"]]
        suma = sub.groupby("COD_FUNCION")["DEVENGADO"].sum()
        ult = sub[sub["MES"] == meses_map[mes_ultimo]].groupby("COD_FUNCION")["DEVENGADO"].sum()
        for cod in sorted(set(ref) | set(suma.index)):
            filas.append({"AÑO": anio, "DEPARTAMENTO": d["departamento"], "COD_FUNCION": cod,
                          "DEV_REF_ANUAL": ref.get(cod, 0.0),
                          "DEV_SUMA_MESES": float(suma.get(cod, 0.0)),
                          "DEV_ULTIMO_MES": float(ult.get(cod, 0.0))})
    ctl = pd.DataFrame(filas)
    if ctl.empty:
        return ctl, "SIN REFERENCIA"
    tol = max(5, len(meses) * 5)
    ctl["DIF_SUMA"] = ctl["DEV_SUMA_MESES"] - ctl["DEV_REF_ANUAL"]
    ctl["DIF_ULTIMO"] = ctl["DEV_ULTIMO_MES"] - ctl["DEV_REF_ANUAL"]
    ok_suma = ctl["DIF_SUMA"].abs().le(tol).all()
    ok_ult = ctl["DIF_ULTIMO"].abs().le(tol).all()
    if ok_suma:
        semantica, col = "MENSUAL (la suma de los meses cuadra con el anual)", "DIF_SUMA"
    elif ok_ult:
        semantica, col = "ACUMULADO (el último mes cuadra con el anual: NO sumar meses)", "DIF_ULTIMO"
    else:
        semantica, col = "NO CUADRA (revisar)", "DIF_SUMA"
    ctl["ESTADO"] = np.where(ctl[col].abs().le(tol), "OK", "REVISAR")
    ctl["TOLERANCIA_S/"] = tol
    ctl["FECHA_EXTRACCION"] = date.today().isoformat()
    return ctl, semantica

print("✅ Configuración lista")

In [ ]:
# ═══ BLOQUE 3 · DIAGNÓSTICO RÁPIDO (requiere el Bloque 2; ~5 consultas) ═══
# Prueba la navegación con Loreto / mes 1 antes de lanzar el año completo.
# Si algo falla, descarga los respuesta_*.html y compártelos.
D_DEPTO, D_MES = "16", "1"

def mostrar(titulo, p, err, archivo):
    print("═" * 78); print(titulo)
    if err:
        print("   ❌", err); return
    print("   URL:", p.url)
    print("   Breadcrumb:")
    for et, dev in p.filtros:
        print(f"      {et[:70]}  ->  {dev:,.0f}")
    print(f"   Filas de detalle: {len(p.filas)}  (primeras 8)")
    for valor, et, dev in p.filas[:8]:
        print(f"      [grp1={valor[:25]}] {et[:60]}  ->  {dev:,.0f}")
    aviso = cuadre(p, titulo)
    print("   " + (f"⚠️ {aviso}" if aviso else "✅ cuadre OK" if p.filas else ""))
    with open(f"/content/{archivo}", "wb") as f:
        f.write(p.html)

p0, e0 = abrir_departamento(AÑO, D_MES, D_DEPTO)
mostrar("1 · Página del departamento", p0, e0, "respuesta_1_departamento.html")
if p0:
    print("   Errores de validación:", validar_departamento(p0, D_DEPTO) or "ninguno")
    print("   Botones:", ", ".join(f"{v[1]}{' (deshab.)' if v[2] else ''}" for v in p0.botones.values()))
    p1, e1 = pulsar(p0, "Función")
    mostrar("2 · Departamento > Función", p1, e1, "respuesta_2_funcion.html")
    p2, e2 = pulsar(p0, "Pliego")
    mostrar("3 · Departamento > Pliego", p2, e2, "respuesta_3_pliego.html")
    if p2 and p2.filas:
        p3, e3 = pulsar(p2, "Ejecutora", p2.filas[0][0])
        mostrar("4 · Pliego > Ejecutora", p3, e3, "respuesta_4_ejecutora.html")
        if p3 and p3.filas:
            p4, e4 = pulsar(p3, "Función", p3.filas[0][0])
            mostrar("5 · Ejecutora > Función", p4, e4, "respuesta_5_ejec_funcion.html")
print("═" * 78)
print("Si los pasos 2 y 5 listan funciones con ✅, ejecuta el Bloque 4.")

In [ ]:
# ═══ BLOQUE 4 · EXTRACCIÓN COMPLETA ═══
drive = conectar_drive(DRIVE_FOLDER_ID) if SUBIR_A_DRIVE else None

# ── 5. BUCLE MES A MES (regiones en secuencia: Loreto -> San Martín -> Ucayali) ─
registros_todos, errores_totales, avisos_totales = [], [], []
archivos_generados, resumen_meses = [], []

print(f"Iniciando extracción {AÑO}: {len(meses)} mes(es) x {len(departamentos)} región(es).\n")

for mes in tqdm(meses, desc="Meses"):
    etiqueta = meses_map[mes]
    regs_mes, errs_mes = [], []
    for d in departamentos:
        regs, err, aviso = extraer(AÑO, mes, d["codigo"])
        regs_mes.extend(regs)
        if err:
            errs_mes.append({"AÑO": AÑO, "MES": etiqueta, "DEPARTAMENTO": d["departamento"], "ERROR": err})
        if aviso:
            avisos_totales.append({"AÑO": AÑO, "MES": etiqueta, "DEPARTAMENTO": d["departamento"], "AVISO": aviso})
        time.sleep(random.uniform(*PAUSA))

    errores_totales.extend(errs_mes)
    registros_todos.extend(regs_mes)
    df_mes = pd.DataFrame(regs_mes)
    if df_mes.empty:
        estado = "SIN DATOS"
    else:
        archivo = os.path.join(ruta_guardado, nombre_archivo_salida(AÑO, [mes]))
        df_mes[COLUMNAS_SALIDA].to_csv(archivo, index=False, encoding="utf-8-sig")
        archivos_generados.append(archivo)
        if SUBIR_A_DRIVE:
            subir_a_drive(drive, archivo, DRIVE_FOLDER_ID)
        estado = "OK" if not errs_mes else "INCOMPLETO (con errores)"
    resumen_meses.append({"MES": etiqueta, "FILAS": len(df_mes), "ERRORES": len(errs_mes), "ESTADO": estado})

print("═" * 70)
print(f"RESUMEN {AÑO}")
display(pd.DataFrame(resumen_meses))
if errores_totales:
    print("⚠️ Consultas con error:")
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(errores_totales))
if avisos_totales:
    print("⚠️ Avisos de cuadre interno (suma del detalle ≠ fila del filtro):")
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(avisos_totales))
if not errores_totales and not avisos_totales:
    print("✅ Sin errores y con cuadre interno en todas las consultas.")

# ── 6. CONTROL ANUAL + CIERRE (guardado, Drive, descarga) ────────────────────
ctl_anual = pd.DataFrame()
df_total = pd.DataFrame(registros_todos)
cubre_anio = (MES_INICIO == 1) and (AÑO < hoy.year or INCLUIR_MES_EN_CURSO)

if CONTROL_ANUAL and df_total.empty:
    print("\nControl anual omitido: no hay datos raspados.")
elif CONTROL_ANUAL and not cubre_anio:
    print("\nℹ️ Control anual omitido: el bucle no cubre el año completo "
          "(MES_INICIO>1 o año en curso con meses cerrados). La vista anual del MEF incluye "
          "el mes en curso; usa INCLUIR_MES_EN_CURSO=True para cuadrarla el mismo día.")
elif CONTROL_ANUAL:
    print("\n🔍 Control anual contra Consulta Amigable…")
    ctl_anual, semantica = control_anual(df_total, AÑO, meses[-1])
    print(f"   Semántica detectada del Devengado por mes: {semantica}")
    if not ctl_anual.empty:
        display(ctl_anual["ESTADO"].value_counts().to_frame("funciones"))
        display(ctl_anual[ctl_anual["ESTADO"] == "REVISAR"])
        display(ctl_anual.groupby("DEPARTAMENTO")[["DEV_REF_ANUAL", "DEV_SUMA_MESES", "DEV_ULTIMO_MES"]].sum())
        archivo_ctl = os.path.join(ruta_guardado, f"CONTROL GobRegional {AÑO}.csv")
        ctl_anual.to_csv(archivo_ctl, index=False, encoding="utf-8-sig")
        archivos_generados.append(archivo_ctl)
        if SUBIR_A_DRIVE:
            subir_a_drive(drive, archivo_ctl, DRIVE_FOLDER_ID)

if SUBIR_A_DRIVE and (errores_totales or avisos_totales):
    log = pd.concat([pd.DataFrame(errores_totales), pd.DataFrame(avisos_totales)], ignore_index=True)
    archivo_log = os.path.join(ruta_guardado, f"LOG GobRegional {AÑO} - {etiqueta_meses(meses)}.csv")
    log.to_csv(archivo_log, index=False, encoding="utf-8-sig")
    subir_a_drive(drive, archivo_log, DRIVE_FOLDER_ID)

if SUBIR_A_DRIVE:
    print(f"\n☁️ DRIVE: {len(drive_subidos)} subido(s), {len(drive_fallidos)} fallido(s)")
    print(f"   https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")

if not archivos_generados:
    print("\n⚠️ No hay archivos para descargar.")
elif DESCARGA_FINAL == "zip":
    from google.colab import files
    ruta_zip = os.path.join(ruta_guardado, nombre_archivo_salida(AÑO, meses).replace(".csv", ".zip"))
    with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for a in archivos_generados:
            z.write(a, arcname=os.path.basename(a))
    files.download(ruta_zip)
elif DESCARGA_FINAL == "individual":
    from google.colab import files
    for a in archivos_generados:
        files.download(a)
else:
    print(f"\n💾 Archivos guardados en {ruta_guardado}")